In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

RAW_PATH = Path('/mnt/data/medquad.csv')
CLEAN_PATH = Path('/mnt/data/medquad_cleaned(1).csv')

raw_df = pd.read_csv(RAW_PATH)
clean_df = pd.read_csv(CLEAN_PATH)

print('Raw shape:', raw_df.shape)
print('Clean shape:', clean_df.shape)
print('Raw columns:', raw_df.columns.tolist())
print('Clean columns:', clean_df.columns.tolist())

In [ ]:
summary = pd.DataFrame({
    'dataset': ['raw_df', 'clean_df'],
    'rows': [len(raw_df), len(clean_df)],
    'columns': [raw_df.shape[1], clean_df.shape[1]],
    'missing_values_total': [raw_df.isna().sum().sum(), clean_df.isna().sum().sum()],
    'full_duplicate_rows': [raw_df.duplicated().sum(), clean_df.duplicated().sum()]
})
summary

In [ ]:
missing_before_after = pd.DataFrame({
    'missing_raw': raw_df.isna().sum(),
    'missing_clean': clean_df.reindex(columns=raw_df.columns).isna().sum()
})
missing_before_after

In [ ]:
raw_df[raw_df['answer'].isna() | raw_df['focus_area'].isna()].head(10)

In [ ]:
duplicate_stats = {
    'full_duplicate_rows_raw': raw_df.duplicated().sum(),
    'full_duplicate_rows_clean': clean_df.duplicated().sum(),
    'duplicate_questions_exact_raw': raw_df['question'].duplicated().sum(),
    'duplicate_questions_case_insensitive_raw': raw_df['question'].astype(str).str.strip().str.lower().duplicated().sum(),
    'duplicate_answers_raw': raw_df['answer'].duplicated().sum(),
}

pd.DataFrame.from_dict(duplicate_stats, orient='index', columns=['count'])

In [ ]:
q_norm = raw_df['question'].astype(str).str.strip().str.lower()
raw_df[q_norm.duplicated(keep=False)].sort_values('question').head(10)

In [ ]:
answer_counts = raw_df['answer'].value_counts(dropna=True).reset_index()
answer_counts.columns = ['answer', 'count']
answer_counts.head(10)

In [ ]:
def has_irregular_spaces(text):
    text = str(text)
    return bool(re.search(r'\s{2,}|', text))

irregular_count = raw_df['answer'].apply(has_irregular_spaces).sum()
top_of_page_count = raw_df['answer'].astype(str).str.contains('Top of page', case=False, na=False).sum()
raw_links_count = raw_df['answer'].astype(str).str.contains(r'https?://|www\.', regex=True, na=False).sum()

pd.DataFrame({
    'issue': ['irregular spaces / newlines / tabs', 'contains Top of page', 'contains raw links'],
    'count': [irregular_count, top_of_page_count, raw_links_count]
})

In [ ]:
def normalize_text(text):
    if pd.isna(text):
        return text
    text = str(text)
    text = re.sub(r'Top of page', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

example_text = raw_df.loc[raw_df['answer'].astype(str).str.contains('Top of page', case=False, na=False), 'answer'].iloc[0]
print('Before:', example_text[:700])
print('After:, normalize_text(example_text)[:700]')

In [ ]:
def is_malformed_question(q):
    q = str(q).strip()
    patterns = [
        r'^What is \(are\)\s*\?$',
        r'^What is\s*\?$',
        r'^What are\s*\?$',
        r'^\?$',
        r'^nan$'
    ]
    return any(re.search(p, q, flags=re.IGNORECASE) for p in patterns)

malformed = raw_df[raw_df['question'].apply(is_malformed_question)]
print('Malformed questions count:', len(malformed))
malformed.head(20)

In [ ]:
raw_df['answer_length_chars'] = raw_df['answer'].fillna('').astype(str).str.len()
short_answers = raw_df[raw_df['answer_length_chars'] <= 20][['question', 'answer', 'source', 'focus_area', 'answer_length_chars']]
short_answers.sort_values('answer_length_chars').head(20)

In [ ]:
clean_df['is_boilerplate'].value_counts(dropna=False).rename('count').to_frame()

In [ ]:
clean_df[clean_df['is_boilerplate'] == True][['doc_id', 'question', 'answer', 'source', 'focus_area']].head(10)

In [ ]:
rag_df_without_boilerplate = clean_df[clean_df['is_boilerplate'] == False].copy()
print('All cleaned rows:', len(clean_df))
print('Rows without boilerplate:', len(rag_df_without_boilerplate))

In [ ]:
clean_df[['doc_id', 'question', 'source', 'focus_area', 'is_boilerplate']].head()

In [ ]:
print('Number of rows:', len(clean_df))
print('Number of unique doc_id:', clean_df['doc_id'].nunique())

clean_df['doc_id'].value_counts().head()